## Analisis de finanzas y riesgo crediticio

#### En quina mesura els clients amb saldos més baixos estan en més risc d'incompliment de crèdit, i com hem d'ajustar les nostres polítiques de crèdit per mitigar aquest risc?

##### librerias

In [34]:
import pandas as pd
import numpy as np


#### Import dataset

In [35]:

# Ruta CSV 
data_path = "../data/BBDD_bank.csv"

# Cargar el dataset
df = pd.read_csv(
    data_path,
    sep=",",
    encoding="utf-8"
)

# Verificar dimensiones iniciales
df.shape


(10643, 18)

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10643 entries, 0 to 10642
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              10643 non-null  int64  
 1   age             10634 non-null  float64
 2   job             10643 non-null  object 
 3   marital         10643 non-null  object 
 4   education       10643 non-null  object 
 5   credit_default  10643 non-null  int64  
 6   balance         10643 non-null  int64  
 7   housing         10643 non-null  int64  
 8   loan            10643 non-null  int64  
 9   contact         10643 non-null  object 
 10  day             10643 non-null  int64  
 11  month           10643 non-null  object 
 12  duration        10643 non-null  int64  
 13  campaign        10643 non-null  int64  
 14  pdays           10643 non-null  int64  
 15  previous        10643 non-null  int64  
 16  poutcome        10643 non-null  object 
 17  deposit         10643 non-null 

In [37]:
df.head()


,id,age,job,marital,education,credit_default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,1,59.0,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,unknown,1
1,2,59.0,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,unknown,1
2,3,56.0,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,unknown,1
3,4,41.0,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,unknown,1
4,5,55.0,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,unknown,1


### 1  Check variables age, housing, job, credit_default y balance

#### 1.1  Check variables age, housing y job

In [38]:
# Verificar rango de edad
df["age"].describe()


count    10634.000000
mean        41.254279
std         12.003685
min         18.000000
25%         32.000000
50%         39.000000
75%         49.000000
max         95.000000
Name: age, dtype: float64

In [39]:
# Verificar valores únicos de housing y loan
df["housing"].value_counts(dropna=False), df["loan"].value_counts(dropna=False)


(housing
 0    5660
 1    4983
 Name: count, dtype: int64,
 loan
 0    9271
 1    1372
 Name: count, dtype: int64)

In [40]:
# Número de categorías en job
df["job"].value_counts()


job
management       2447
blue-collar      1830
technician       1737
admin.           1280
services          881
retired           755
self-employed     388
student           354
unemployed        338
entrepreneur      308
housemaid         257
unknown            68
Name: count, dtype: int64

#### 1.2 Análisis de la variable credit_default y balance

In [41]:
df["credit_default"].value_counts(), df["credit_default"].value_counts(normalize=True) * 100


(credit_default
 0    10488
 1      155
 Name: count, dtype: int64,
 credit_default
 0    98.543644
 1     1.456356
 Name: proportion, dtype: float64)

In [42]:
df["balance"].describe()

count    10643.000000
mean      1536.690595
std       3223.277326
min      -6847.000000
25%        127.000000
50%        558.000000
75%       1729.500000
max      81204.000000
Name: balance, dtype: float64

In [45]:
# Estadísticas descriptivas ampliadas de balance
df["balance"].describe(percentiles=[0.01, 0.05, 0.10, 0.90, 0.95, 0.99])


count    10643.000000
mean      1536.690595
std       3223.277326
min      -6847.000000
1%        -516.480000
5%         -48.800000
10%          0.000000
50%        558.000000
90%       3911.200000
95%       6004.900000
99%      13101.540000
max      81204.000000
Name: balance, dtype: float64

In [46]:
# Proporción de clientes por tipo de balance
balance_area_summary = pd.Series({
    "negativo": (df["balance"] < 0).mean() * 100,
    "cero": (df["balance"] == 0).mean() * 100,
    "positivo": (df["balance"] > 0).mean() * 100
}).round(2)

balance_area_summary


negativo     6.07
cero         6.83
positivo    87.10
dtype: float64

In [ ]:
# Estadísticas solo balances positivos
df.loc[df["balance"] > 0, "balance"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
)


count     9270.000000
mean      1786.734412
std       3380.539232
min          1.000000
10%         85.000000
25%        262.000000
50%        740.000000
75%       2037.000000
90%       4333.000000
max      81204.000000
Name: balance, dtype: float64

### 2. Creacion Columna de segmentacion de balance



In [51]:

# Crear columna de segmentación de balance
df["balance_segment"] = np.nan

# Segmento negativo
df.loc[df["balance"] < 0, "balance_segment"] = "negativo"

# Segmento cero + positivo
non_negative_mask = df["balance"] >= 0

# Subsegmentar cero y positivos en tres grupos
df.loc[non_negative_mask, "balance_segment"] = pd.qcut(
    df.loc[non_negative_mask, "balance"],
    q=3,
    labels=["bajo", "medio", "alto"]
)


C:\Users\edo\AppData\Local\Temp\ipykernel_13284\2051878538.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'negativo' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["balance"] < 0, "balance_segment"] = "negativo"


In [52]:
# Verificar distribución de los segmentos
df["balance_segment"].value_counts()


balance_segment
bajo        3338
alto        3333
medio       3326
negativo     646
Name: count, dtype: int64

### 3. relacion balance_segment y riesgo

####  3.1 % default por balance_segment

In [ ]:
# Tamaño del grupo balance_segment y porcentaje de default
balance_segment_summary = (
    df
    .groupby("balance_segment")
    .agg(
        total_clientes=("credit_default", "count"),
        porcentaje_default=("credit_default", lambda x: round(x.mean() * 100, 2))
    )
    .sort_values(by="porcentaje_default", ascending=False)
)

balance_segment_summary


,total_clientes,porcentaje_default
balance_segment,,
negativo,646,11.46
bajo,3338,1.89
medio,3326,0.45
alto,3333,0.09


#### 3.2 balance vs loan y housing

In [60]:
# Tabla balance_segment x loan (% credit_default)
default_by_balance_loan = (
    df
    .groupby(["balance_segment", "loan"])["credit_default"]
    .mean()
    .mul(100)
    .round(2)
    .unstack()
)

default_by_balance_loan


loan,0,1
balance_segment,,
alto,0.10,0.00
bajo,1.40,4.70
medio,0.38,0.94
negativo,10.63,13.24


In [61]:
# Tabla balance_segment x housing (% credit_default)
default_by_balance_housing = (
    df
    .groupby(["balance_segment", "housing"])["credit_default"]
    .mean()
    .mul(100)
    .round(2)
    .unstack()
)

default_by_balance_housing


housing,0,1
balance_segment,,
alto,0.00,0.23
bajo,2.52,1.19
medio,0.40,0.51
negativo,14.63,10.37


In [59]:
df["balance_segment"].value_counts()


balance_segment
bajo        3338
alto        3333
medio       3326
negativo     646
Name: count, dtype: int64

###  3.3 balance vs age y job

# poner en seccion correcta?

In [62]:
# Crear grupos de edad
df["age_group"] = pd.cut(
    df["age"],
    bins=[18, 30, 45, 60, 100],
    labels=["18-30", "31-45", "46-60", "60+"]
)


In [63]:
# Tabla balance_segment x age_group (% credit_default)
default_by_balance_age = (
    df
    .groupby(["balance_segment", "age_group"])["credit_default"]
    .mean()
    .mul(100)
    .round(2)
    .unstack()
)

default_by_balance_age


C:\Users\edo\AppData\Local\Temp\ipykernel_13284\1609569030.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["balance_segment", "age_group"])["credit_default"]


age_group,18-30,31-45,46-60,60+
balance_segment,,,,
alto,0.21,0.00,0.20,0.00
bajo,2.42,1.72,2.09,0.00
medio,0.30,0.43,0.58,0.61
negativo,11.02,10.95,12.85,0.00


In [64]:
# Seleccionar los 3 jobs más frecuentes
top_jobs = df["job"].value_counts().head(3).index
df_top_jobs = df[df["job"].isin(top_jobs)]


In [65]:
# Tabla balance_segment x job (% credit_default)
default_by_balance_job = (
    df_top_jobs
    .groupby(["balance_segment", "job"])["credit_default"]
    .mean()
    .mul(100)
    .round(2)
    .unstack()
)

default_by_balance_job


job,blue-collar,management,technician
balance_segment,,,
alto,0.00,0.23,0.00
bajo,1.57,2.43,2.22
medio,0.85,0.55,0.36
negativo,13.27,9.28,12.75


### 4. Insights y políticas de crédito

### 5. Limitaciones